# Chest X-Ray Disease Classification — Google Colab Training Template

This notebook mirrors `notebooks/kaggle_training_template.ipynb` so that switching between
Kaggle and Colab does **not** change the experiment-tracking standard. It is a template,
not a finished model -- **TODO** sections are dataset/model-specific.

See `docs/experiment_policy.md` and `docs/wandb_setup.md` for the policy this notebook follows.

## 1. GPU / runtime verification

Runtime -> Change runtime type -> GPU, before running this cell.

In [ ]:
import platform
print('Python:', platform.python_version())

try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except ImportError:
    print('PyTorch not found -- Colab GPU runtimes normally ship it preinstalled.')

## 2. Clone / import project

**TODO:** replace the URL below with this repository's clone URL
(or mount Drive if you keep a persistent local clone there -- see step 4).

In [ ]:
import os

REPO_URL = 'https://github.com/REPLACE_ME/Chest-X-ray-Disease-Detection.git'  # TODO
REPO_DIR = '/content/Chest-X-ray-Disease-Detection'

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR

## 3. Install requirements

PyTorch is already provided by the Colab runtime -- install only the tracking/config libraries.

In [ ]:
%pip install -q wandb PyYAML

## 4. (Optional) Mount Google Drive

Useful for persisting checkpoints/data across sessions -- Colab's local disk is wiped
when the runtime recycles.

In [ ]:
MOUNT_DRIVE = False  # TODO: set True if you want to persist checkpoints/data to Drive

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

## 5. W&B login

Prefer `wandb.login()` (opens a secure prompt) or a Colab secret over pasting a key
into a cell. See `docs/wandb_setup.md`.

In [ ]:
import wandb

try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    print('W&B API key loaded from Colab secret.')
except Exception:
    print('No Colab secret found -- falling back to interactive login.')
    wandb.login()

## 6. Import project code and load YAML configuration

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(REPO_DIR)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import (
    load_config,
    set_seed,
    create_generator,
    seed_worker,
    initialize_wandb,
    log_metrics,
    log_summary_metrics,
    finish_run,
    generate_run_name,
    BestCheckpointSaver,
)

config = load_config(PROJECT_ROOT / 'configs' / 'baseline.yaml')
import json; print(json.dumps(config, indent=2))

## 7. Set random seed

In [ ]:
set_seed(config['experiment']['seed'])
generator = create_generator(config['experiment']['seed'])

## 8. Dataset location placeholder

**TODO:** point this at wherever the dataset lives for this run (Drive, a downloaded
Kaggle dataset, etc.).

In [ ]:
DATASET_DIR = Path('/content/REPLACE_ME')  # TODO
OUTPUT_DIR = Path('/content/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 9. Fixed split manifest

**Policy:** reuse the same split manifest as every other model -- never re-split per run.
See `docs/experiment_policy.md`.

In [ ]:
# TODO: load the split manifest, e.g.:
# import pandas as pd
# split_manifest = pd.read_csv(DATASET_DIR / f"splits/{config['dataset']['split_version']}.csv")
# train_df = split_manifest[split_manifest['split'] == 'train']
# val_df = split_manifest[split_manifest['split'] == 'val']
# test_df = split_manifest[split_manifest['split'] == 'test']

## 10. Model / training placeholders

**TODO:** replace with the real model, optimizer, and training/validation loops.

In [ ]:
# TODO: model
# import timm
# model = timm.create_model(config['model']['name'], pretrained=config['model']['pretrained'],
#                            num_classes=config['model']['num_classes']).to('cuda')

# TODO: optimizer
# import torch.optim as optim
# optimizer = optim.AdamW(model.parameters(), lr=config['training']['learning_rate'],
#                          weight_decay=config['training']['weight_decay'])

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    for batch in loader:  # TODO: unpack (inputs, targets) once the Dataset exists
        raise NotImplementedError('Fill in the training step for this model.')
    return running_loss / max(len(loader), 1)

def evaluate(model, loader, criterion, device):
    model.eval()
    # TODO: compute real val_loss/metrics -- do not fabricate values
    return {
        'val_loss': None, 'val_accuracy': None, 'val_precision': None,
        'val_recall': None, 'val_f1': None, 'val_auroc': None,
    }

## 11. W&B tracking

In [ ]:
run_name = generate_run_name(
    config['model']['name'], config['experiment']['name'], config['experiment']['seed']
)
print('Run name:', run_name)
run = initialize_wandb(config, run_name=run_name, mode=None)

## 12. Checkpoint saving

In [ ]:
checkpoint_saver = BestCheckpointSaver(
    run_name=run_name,
    monitor=config['checkpoint']['monitor'],
    mode=config['checkpoint']['mode'],
    checkpoint_dir=OUTPUT_DIR / 'checkpoints',
)

# TODO: uncomment once train_one_epoch/evaluate/model/optimizer are implemented
# device = 'cuda' if torch.cuda.is_available() else 'cpu'
# for epoch in range(config['training']['epochs']):
#     train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
#     val_metrics = evaluate(model, val_loader, criterion, device)
#     log_metrics({
#         'epoch': epoch + 1,
#         'train_loss': train_loss,
#         'learning_rate': optimizer.param_groups[0]['lr'],
#         **val_metrics,
#     })
#     improved = checkpoint_saver.step(
#         epoch=epoch + 1,
#         metric_value=val_metrics[config['checkpoint']['monitor']],
#         model=model, optimizer=optimizer, config=config,
#     )

## 13. Finalization

**TODO:** run final evaluation on the held-out test split before finishing the run.

In [ ]:
log_summary_metrics(checkpoint_saver.summary())
finish_run()